# nanoHUB-QE Tutorial

This notebook shows how to use `nanohubqe` on nanoHUB, including loading Quantum ESPRESSO with the `use` command behavior.

In [ ]:
import nanohubqe
from nanohubqe import __version__
print('nanohubqe version:', __version__)

## 1. Load Quantum ESPRESSO on nanoHUB

On nanoHUB, load the QE module before running workflows.
You can use either the Python function or the Jupyter magic (`%use`, `%use_qe`).

In [ ]:
from nanohubqe import list_available_modules, load_quantum_espresso

qe_like = list_available_modules('qe')
espresso_like = list_available_modules('espresso')
print('QE-like modules:', qe_like[:10])
print('ESPRESSO-like modules:', espresso_like[:10])

try:
    loaded = load_quantum_espresso()
    print('Loaded module:', loaded)
except Exception as exc:
    print('Auto-detect failed:', exc)
    print("Call load_quantum_espresso('your-module-name') if needed.")


## 2. Build a nanoHUB-style Si SCF + DOS + Bands workflow

In [ ]:
from shutil import which
from nanohubqe import QERunner, silicon_bands_dos_reference_workflow

workflow = silicon_bands_dos_reference_workflow(
    a=5.43,
    ecutwfc=16.0,
    ecutrho=96.0,
    nbnd=8,
    scf_k_points=(4, 4, 4, 0, 0, 0),
    dos_emin=-6.0,
    dos_emax=10.0,
    dos_deltae=0.1,
    include_plotband=False,
)
runner = QERunner(default_backend='local')

required = ['pw.x', 'dos.x', 'bands.x']
missing = [exe for exe in required if which(exe) is None]
dry_run = bool(missing)
if dry_run:
    print('Missing executables:', ', '.join(missing))
    print('Falling back to dry_run=True (commands only).')
else:
    print('Running workflow with real QE executables...')

results = runner.run_workflow(workflow, workdir='runs/tutorial-si', dry_run=dry_run)
for step, result in results.items():
    print(step, 'rc=', result.returncode, 'ok=', result.ok)


## 3. Inspect expected/discovered outputs and run records

In [ ]:
from pathlib import Path

record_path = Path('runs/tutorial-si/workflow_outputs.json')
print('JSON record:', record_path, 'exists=', record_path.exists())

dos_result = results.get('dos')
if dos_result is None:
    print('DOS step not available (workflow may have stopped early).')
else:
    print('Expected:', dos_result.expected_outputs)
    print('Discovered:', [p.name for p in dos_result.discovered_outputs])


## 4. Parse and plot outputs

When real run outputs are available (`dry_run=False`), parse and plot with either backend.

In [ ]:
from pathlib import Path
from nanohubqe import plot_dos

dos_file = Path('runs/tutorial-si/qe.dos')
if dos_file.exists():
    fig = plot_dos(dos_file, backend='plotly')
    fig
else:
    print('No DOS file found yet. Re-run the workflow cell after loading QE modules.')


## 5. Jupyter magics

If IPython magics are available, you can also do:

```python
%use quantum-espresso-7.x
%use_qe
```